# Phase 1 — Diagnostic Analysis: MEF-LSTM Itajaí-Açu

**Goal**: Quantify exactly *what* is failing and *where* before retraining, so experiments 06–09 target the right bottlenecks.

## Sections
1. [Residual analysis by discharge quantile](#sec1) — at what flow threshold does the model diverge?
2. [Antecedent precipitation lag correlation](#sec2) — would extending `seq_length` help the 2011 event?
3. [CHIRPS precipitation bias on extreme events](#sec3) — how wrong is the input forcing?
4. [Residual temporal structure & lead-time degradation](#sec4) — seasonal patterns and ACF of errors

**Reference model**: exp03_nse_loss (NSE t+7 = 0.474, best so far)

In [ ]:
from __future__ import annotations
import sys
from pathlib import Path

ROOT = Path().resolve().parents[1]
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.gridspec import GridSpec
from scipy import stats

plt.rcParams.update({
    'figure.dpi': 120,
    'font.size': 11,
    'axes.titlesize': 12,
    'axes.labelsize': 11,
    'legend.fontsize': 9,
})

ZARR_EXP03 = ROOT / 'models/experiments/exp03_nse_loss/exp03_nse_loss_0906_112336/test/model_epoch030/test_results.zarr'
ZARR_BASELINE = ROOT / 'models/experiments/mef_lstm_itajai_baseline_0906_083244/test/model_epoch015/test_results.zarr'
CSV_TS   = ROOT / 'data/processed/Caravan-nc/timeseries/csv/itajai/itajai_83500000.csv'
CHIRPS_P = ROOT / 'data/raw/chirps/chirps_itajai_mean_all.parquet'
ANA_P    = ROOT / 'data/raw/ana/83500000_vazao_raw.parquet'
FIGURES  = ROOT / 'reports/figures'
FIGURES.mkdir(parents=True, exist_ok=True)

In [ ]:
# ── Load exp03 zarr (all 8 lead times) ───────────────────────────────────────
ds = xr.open_zarr(str(ZARR_EXP03), consolidated=False).compute()
basin = str(ds.basin.values[0])

dates = pd.DatetimeIndex(ds.date.values)
obs   = ds['streamflow_obs'].sel(basin=basin, freq='1D').values.squeeze()  # (6210, 8)
sim   = ds['streamflow_sim'].sel(basin=basin, freq='1D').values.squeeze()  # (6210, 8)

# Convenience arrays for most-used lead times
obs_t0 = obs[:, 0]   # t+0: same-day (hindcast check)
sim_t0 = sim[:, 0]
obs_t7 = obs[:, 7]   # t+7: main forecast target
sim_t7 = sim[:, 7]
err_t7 = sim_t7 - obs_t7   # positive = overestimate

# ── Load processed timeseries CSV ────────────────────────────────────────────
ts = pd.read_csv(CSV_TS, parse_dates=['date'], index_col='date')
ts_test = ts.loc['2008-01-01':'2024-12-31']

# ── Load raw CHIRPS + ANA ─────────────────────────────────────────────────────
chirps = pd.read_parquet(CHIRPS_P).rename(columns={'precip_mm_chirps': 'precip'})
chirps.index = pd.to_datetime(chirps.index)
ana    = pd.read_parquet(ANA_P).rename(columns={'value': 'streamflow'})
ana.index = pd.to_datetime(ana.index)

print(f'Test period: {dates[0].date()} → {dates[-1].date()}  ({len(dates)} days)')
print(f'obs_t7 range: {np.nanmin(obs_t7):.1f} – {np.nanmax(obs_t7):.1f} m³/s')
print(f'sim_t7 range: {np.nanmin(sim_t7):.1f} – {np.nanmax(sim_t7):.1f} m³/s')

---
## 1. Residual Analysis by Discharge Quantile <a id='sec1'></a>

**Question**: Does model error grow linearly above 600 m³/s (training max), or is there a sudden cliff?  
**Metric**: MAE, relative bias, and standard deviation per discharge quantile bin.

In [ ]:
# ── Quantile-binned error analysis ───────────────────────────────────────────
mask_valid = ~(np.isnan(obs_t7) | np.isnan(sim_t7))
o_v = obs_t7[mask_valid]
s_v = sim_t7[mask_valid]
e_v = s_v - o_v

# Bin edges at key quantiles + explicit thresholds for alert levels
bin_edges = np.unique(np.concatenate([
    np.percentile(o_v, [0, 10, 25, 50, 75, 90, 95, 99, 100]),
    [500, 1000, 1200, 2000]
]))
bin_edges = bin_edges[bin_edges <= o_v.max()]

bins = pd.cut(o_v, bins=bin_edges, include_lowest=True)
df_bins = pd.DataFrame({'obs': o_v, 'sim': s_v, 'err': e_v, 'bin': bins})

stats_df = df_bins.groupby('bin', observed=True).agg(
    count=('obs', 'count'),
    obs_mean=('obs', 'mean'),
    mae=('err', lambda x: np.abs(x).mean()),
    bias=('err', 'mean'),
    rel_bias=('err', lambda x: x.mean() / df_bins.loc[x.index, 'obs'].mean() * 100),
    rmse=('err', lambda x: np.sqrt((x**2).mean())),
).dropna()

print(stats_df.to_string(float_format='{:.1f}'.format))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Residual Analysis by Discharge Quantile — exp03 (t+7)', fontweight='bold')

x = stats_df['obs_mean'].values

# Panel 1: MAE and RMSE
ax = axes[0]
ax.plot(x, stats_df['mae'], 'o-', color='steelblue', label='MAE')
ax.plot(x, stats_df['rmse'], 's--', color='tomato', label='RMSE')
ax.axvline(600, color='gray', lw=1, ls=':', label='~Training max (~600 m³/s)')
ax.axvline(1200, color='orange', lw=1, ls=':', label='Pré-alerta (1200 m³/s)')
ax.set_xlabel('Observed flow bin mean (m³/s)')
ax.set_ylabel('Error (m³/s)')
ax.set_title('MAE & RMSE by flow bin')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Panel 2: Relative bias (%)
ax = axes[1]
colors = ['tomato' if v < 0 else 'steelblue' for v in stats_df['rel_bias']]
ax.bar(range(len(stats_df)), stats_df['rel_bias'], color=colors, edgecolor='gray', lw=0.5)
ax.axhline(0, color='black', lw=0.8)
ax.set_xticks(range(len(stats_df)))
ax.set_xticklabels([f'{v:.0f}' for v in x], rotation=45, ha='right', fontsize=8)
ax.set_xlabel('Flow bin mean (m³/s)')
ax.set_ylabel('Relative bias (%)')
ax.set_title('Relative bias by flow bin\n(negative = underestimate)')
ax.grid(True, axis='y', alpha=0.3)

# Panel 3: Scatter obs vs sim, coloured by quantile
ax = axes[2]
sc = ax.scatter(o_v, s_v, c=o_v, cmap='YlOrRd', s=4, alpha=0.4, norm=plt.Normalize(0, 2000))
lim = max(o_v.max(), s_v.max()) * 1.05
ax.plot([0, lim], [0, lim], 'k--', lw=1, label='1:1 line')
ax.set_xlabel('Observed (m³/s)')
ax.set_ylabel('Simulated t+7 (m³/s)')
ax.set_title('Observed vs Simulated (t+7)')
ax.set_xlim(0, lim); ax.set_ylim(0, lim)
plt.colorbar(sc, ax=ax, label='Observed flow (m³/s)')
ax.grid(True, alpha=0.3)

plt.tight_layout()
fig.savefig(FIGURES / 'diag1_residuals_by_quantile.png', dpi=150, bbox_inches='tight')
plt.show()

# Key finding
threshold_idx = np.searchsorted(x, 600)
bias_below = stats_df['rel_bias'].iloc[:threshold_idx].mean()
bias_above = stats_df['rel_bias'].iloc[threshold_idx:].mean()
print(f'\n=== KEY FINDING ===')
print(f'Average relative bias BELOW ~600 m³/s: {bias_below:.1f}%')
print(f'Average relative bias ABOVE ~600 m³/s: {bias_above:.1f}%')
print(f'→ Gap of {abs(bias_above - bias_below):.1f} pp shows '
      f'{"a sudden cliff" if abs(bias_above - bias_below) > 20 else "gradual degradation"} above training max.')

---
## 2. Antecedent Precipitation Lag Correlation <a id='sec2'></a>

**Question**: Does prediction error correlate with basin wetness *before* the event window?  
If yes, at what lag? This tells us the minimum `seq_length` needed for exp06.

**Method**: For each high-flow day (obs > 500 m³/s), compute cumulative precipitation at
look-back windows of 7, 14, 21, 30, 45, 60 days. Correlate with absolute prediction error.

In [ ]:
# ── Build aligned dataframe: date, obs, sim, error, antecedent precip ────────
df_all = pd.DataFrame({
    'obs':  obs_t7,
    'sim':  sim_t7,
    'err':  err_t7,
    'abserr': np.abs(err_t7),
    'relerr': np.where(obs_t7 > 0, np.abs(err_t7) / obs_t7, np.nan),
}, index=dates)

# Align CHIRPS to the same date index
chirps_aligned = chirps['precip'].reindex(dates).fillna(0)

# Compute rolling cumulative precipitation at multiple look-back windows
lags = [7, 14, 21, 30, 45, 60]
for lag in lags:
    df_all[f'precip_cum_{lag}d'] = chirps_aligned.rolling(lag, min_periods=lag).sum().values

# Focus on high-flow days (obs > 500 m³/s) for correlation analysis
df_high = df_all[df_all['obs'] > 500].dropna()
print(f'High-flow days (obs > 500 m³/s): {len(df_high)}')

# Pearson correlation: antecedent precip vs absolute error
corrs = {}
for lag in lags:
    col = f'precip_cum_{lag}d'
    r, p = stats.pearsonr(df_high[col].dropna(), df_high.loc[df_high[col].notna(), 'abserr'])
    corrs[lag] = {'r': r, 'p': p}
    print(f'Lag {lag:2d}d  r={r:.3f}  p={p:.4f}{"  **" if p < 0.01 else ""}')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Antecedent Precipitation Lag Correlation — High-flow days (obs > 500 m³/s)',
             fontweight='bold')

# Panel 1: Correlation coefficients by lag
ax = axes[0, 0]
r_vals = [corrs[lag]['r'] for lag in lags]
p_vals = [corrs[lag]['p'] for lag in lags]
bars = ax.bar(lags, r_vals, color=['tomato' if r > 0 else 'steelblue' for r in r_vals],
              edgecolor='gray', lw=0.5)
for lag, r, p in zip(lags, r_vals, p_vals):
    marker = '**' if p < 0.01 else ('*' if p < 0.05 else '')
    ax.text(lag, r + 0.01 * np.sign(r), f'{r:.2f}{marker}', ha='center', fontsize=8)
ax.axhline(0, color='black', lw=0.8)
ax.set_xlabel('Look-back window (days)')
ax.set_ylabel('Pearson r (antecedent precip vs |error|)')
ax.set_title('Correlation: antecedent precip → prediction error\n** p<0.01, * p<0.05')
ax.set_xticks(lags)
ax.grid(True, axis='y', alpha=0.3)

# Panel 2: Scatter for the lag with highest correlation
best_lag = lags[np.argmax(np.abs(r_vals))]
col_best = f'precip_cum_{best_lag}d'
ax = axes[0, 1]
ax.scatter(df_high[col_best], df_high['abserr'], alpha=0.5, s=15, color='steelblue')
# Regression line
m, b = np.polyfit(df_high[col_best].dropna(),
                  df_high.loc[df_high[col_best].notna(), 'abserr'], 1)
xfit = np.linspace(df_high[col_best].min(), df_high[col_best].max(), 100)
ax.plot(xfit, m * xfit + b, 'r-', lw=1.5, label=f'OLS fit (r={corrs[best_lag]["r"]:.2f})')
ax.set_xlabel(f'Cumulative precip last {best_lag} days (mm)')
ax.set_ylabel('|Error| t+7 (m³/s)')
ax.set_title(f'Strongest lag: {best_lag} days')
ax.legend()
ax.grid(True, alpha=0.3)

# Panel 3: Sep 2011 — antecedent 60-day precip vs historical distribution
ax = axes[1, 0]
peak_2011 = pd.Timestamp('2011-09-11')   # approximate observed peak
# For each year, cumulative precip in the 60 days ending on same calendar DOY
doy_2011 = peak_2011.dayofyear
cum60_per_year = {}
for yr in range(1996, 2025):
    try:
        end   = pd.Timestamp(f'{yr}-01-01') + pd.Timedelta(days=doy_2011 - 1)
        start = end - pd.Timedelta(days=59)
        val   = chirps.loc[start:end, 'precip'].sum()
        if not np.isnan(val):
            cum60_per_year[yr] = float(val)
    except:
        pass

years = list(cum60_per_year.keys())
vals  = [cum60_per_year[y] for y in years]
color_2011 = ['tomato' if y == 2011 else 'steelblue' for y in years]
ax.bar(years, vals, color=color_2011, edgecolor='gray', lw=0.4)
ax.axhline(np.median(vals), color='black', lw=1, ls='--', label=f'Median {np.median(vals):.0f} mm')
ax.set_xlabel('Year')
ax.set_ylabel('Cumulative precip 60d before peak (mm)')
ax.set_title('Basin wetness at Sep/2011 peak vs historical\n(60-day antecedent precip)')
ax.legend()
ax.grid(True, axis='y', alpha=0.3)

# Panel 4: Same for all high-flow peaks > 800 m³/s
ax = axes[1, 1]
df_extreme = df_all[(df_all['obs'] > 800)].dropna(subset=['precip_cum_60d', 'abserr'])
sc = ax.scatter(df_extreme['precip_cum_60d'], df_extreme['abserr'],
                c=df_extreme['obs'], cmap='YlOrRd', s=40, edgecolors='gray', lw=0.3,
                norm=plt.Normalize(800, 2600), zorder=5)
# Highlight 2011
mask_2011 = (df_extreme.index >= '2011-09-01') & (df_extreme.index <= '2011-09-30')
if mask_2011.any():
    ax.scatter(df_extreme.loc[mask_2011, 'precip_cum_60d'],
               df_extreme.loc[mask_2011, 'abserr'],
               color='red', s=120, marker='*', zorder=10, label='Set/2011')
plt.colorbar(sc, ax=ax, label='Observed flow (m³/s)')
ax.set_xlabel('Cumulative precip last 60 days (mm)')
ax.set_ylabel('|Error| t+7 (m³/s)')
ax.set_title('Extreme events (obs > 800 m³/s):\nerror vs antecedent wetness')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
fig.savefig(FIGURES / 'diag2_antecedent_precip_lag.png', dpi=150, bbox_inches='tight')
plt.show()

val_2011 = cum60_per_year.get(2011, np.nan)
pct_2011 = np.mean(np.array(vals) <= val_2011) * 100
print(f'\n=== KEY FINDING ===')
print(f'Best lag (highest |r|): {best_lag} days  (r={corrs[best_lag]["r"]:.3f})')
print(f'Sep 2011 antecedent 60-day precip: {val_2011:.0f} mm  (percentile {pct_2011:.0f}%)')
print(f'Median historical 60d precip: {np.median(vals):.0f} mm')
print(f'→ Extending seq_length to ≥{best_lag} days is recommended for exp06.')

---
## 3. CHIRPS Precipitation Bias on Extreme Events <a id='sec3'></a>

**Question**: For the events where the model fails most (high-flow days), is CHIRPS systematically
underestimating the precipitation input?

**Method**: On extreme event days, compare CHIRPS basin-average precip to what the water balance
implies (ΔQ / basin_area × recession factor). Also compare pre-peak CHIRPS accumulation across years.

In [ ]:
# ── CHIRPS intensity distribution on high-flow vs normal days ─────────────────
# Align CHIRPS precip to test-period dates
precip_test = chirps.loc['2008-01-01':'2024-12-31', 'precip'].reindex(dates).fillna(0).values

mask_high  = obs_t7 > 800    # extreme
mask_med   = (obs_t7 > 300) & (obs_t7 <= 800)  # elevated
mask_low   = obs_t7 <= 300   # normal

p_high = precip_test[mask_high & ~np.isnan(obs_t7)]
p_med  = precip_test[mask_med  & ~np.isnan(obs_t7)]
p_low  = precip_test[mask_low  & ~np.isnan(obs_t7)]

print('CHIRPS daily precip statistics by flow regime:')
for label, arr in [('Normal (Q≤300)', p_low), ('Elevated (300–800)', p_med), ('Extreme (Q>800)', p_high)]:
    print(f'  {label:25s}  mean={np.mean(arr):.2f}  median={np.median(arr):.2f}  '
          f'p95={np.percentile(arr, 95):.1f}  p99={np.percentile(arr, 99):.1f}  n={len(arr)}')

In [ ]:
# ── Pre-peak precipitation windows: 2008 vs 2011 vs average years ─────────────
EVENT_PEAKS = {
    'Nov/2008': '2008-11-23',
    'Set/2011': '2011-09-11',
    'Jan/2022': '2022-01-17',
}

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('CHIRPS Precipitation: 14-day Pre-peak Window — Events vs Historical',
             fontweight='bold')

for ax, (ev_name, peak_date_str) in zip(axes, EVENT_PEAKS.items()):
    peak_dt = pd.Timestamp(peak_date_str)
    window_days = 21

    # Event precip window
    p_event = chirps.loc[peak_dt - pd.Timedelta(days=window_days-1):peak_dt, 'precip']

    # Historical windows for same calendar period across all years
    hist_windows = []
    for yr in range(1996, 2025):
        if yr == peak_dt.year:
            continue
        try:
            pk  = peak_dt.replace(year=yr)
            win = chirps.loc[pk - pd.Timedelta(days=window_days-1):pk, 'precip'].values
            if len(win) == window_days:
                hist_windows.append(win)
        except:
            pass
    hist_arr = np.array(hist_windows)   # (n_years, window_days)
    hist_mean = hist_arr.mean(axis=0)
    hist_p10  = np.percentile(hist_arr, 10, axis=0)
    hist_p90  = np.percentile(hist_arr, 90, axis=0)

    x = np.arange(-window_days + 1, 1)  # days before peak

    ax.fill_between(x, hist_p10, hist_p90, alpha=0.2, color='steelblue', label='Hist. P10–P90')
    ax.plot(x, hist_mean, color='steelblue', lw=1.5, ls='--', label='Hist. mean')
    ax.bar(x, p_event.values, color='tomato', alpha=0.7, width=0.8, label=f'{ev_name} CHIRPS')

    ax.axvline(0, color='black', lw=1, ls=':', alpha=0.6)
    ax.set_xlabel('Days before observed peak')
    ax.set_ylabel('Precip (mm/day)')
    ax.set_title(f'{ev_name}\ncum={p_event.sum():.0f} mm  (hist. mean {hist_mean.sum():.0f} mm)')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
fig.savefig(FIGURES / 'diag3_chirps_bias_events.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Implied rainfall from streamflow rise (rough water balance check) ──────────
# For each flood event, compute ΔQ and compare to CHIRPS accumulation
# Basin area: ~15,000 km² → conversion factor to mm
BASIN_AREA_KM2 = 15_000
M3S_TO_MM_DAY  = 86400 / (BASIN_AREA_KM2 * 1e6) * 1000  # m³/s → mm/day over basin

ana_test = ana.loc['2008-01-01':'2024-12-31', 'streamflow'].reindex(dates).interpolate()

print('Water balance check (rough) — peak events:')
print(f'  Conversion: 1 m³/s ≈ {M3S_TO_MM_DAY*1:.4f} mm/day over {BASIN_AREA_KM2:,} km²\n')
print(f'{"Event":12s}  {"Q_base":>8s}  {"Q_peak":>8s}  {"ΔQ (m³/s)":>12s}  '
      f'{"ΔQ (mm)":>9s}  {"CHIRPS 14d (mm)":>16s}  {"Ratio":>7s}')
print('-' * 80)

for ev_name, peak_date_str in EVENT_PEAKS.items():
    peak_dt = pd.Timestamp(peak_date_str)
    base_dt = peak_dt - pd.Timedelta(days=14)
    try:
        q_base   = float(ana_test.loc[base_dt])
        q_peak   = float(ana_test.loc[peak_dt])
        dq_m3s   = max(q_peak - q_base, 0)
        dq_mm    = dq_m3s * M3S_TO_MM_DAY * 14  # rough: rise over 14 days
        chirps14 = float(chirps.loc[base_dt:peak_dt, 'precip'].sum())
        ratio    = chirps14 / dq_mm if dq_mm > 0 else np.nan
        print(f'{ev_name:12s}  {q_base:8.0f}  {q_peak:8.0f}  {dq_m3s:12.0f}  '
              f'{dq_mm:9.1f}  {chirps14:16.1f}  {ratio:7.2f}')
    except Exception as e:
        print(f'{ev_name:12s}  error: {e}')

print()
print('Note: ratio < 1 → CHIRPS precip alone cannot explain the runoff (soil saturation + ET')  
print('      ratio > 1 → CHIRPS overestimates OR baseflow dominates the rise')
print('      For Sep/2011, expect ratio << 1 (underestimated precip + saturated basin)')

---
## 4. Residual Temporal Structure & Lead-time Degradation <a id='sec4'></a>

**Questions**:  
- Do errors have seasonal structure (wet season bias)?  
- How does NSE degrade from t+0 → t+7?  
- Are residuals autocorrelated (systematic timing errors vs random noise)?

In [ ]:
# ── NSE degradation by lead time ─────────────────────────────────────────────
def nse(o, s):
    mask = ~(np.isnan(o) | np.isnan(s))
    o, s = o[mask], s[mask]
    return 1 - np.sum((o - s)**2) / np.sum((o - o.mean())**2)

lead_nse  = [nse(obs[:, t], sim[:, t]) for t in range(8)]
lead_mae  = [np.nanmean(np.abs(sim[:, t] - obs[:, t])) for t in range(8)]
lead_bias = [np.nanmean(sim[:, t] - obs[:, t]) for t in range(8)]

print('Lead-time performance degradation:')
print(f'{"Lead":>6s}  {"NSE":>8s}  {"MAE (m³/s)":>12s}  {"Bias (m³/s)":>13s}')
for t in range(8):
    print(f't+{t:1d}     {lead_nse[t]:8.3f}  {lead_mae[t]:12.1f}  {lead_bias[t]:13.1f}')

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf

fig = plt.figure(figsize=(16, 12))
gs  = GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)
fig.suptitle('Residual Temporal Structure — exp03 (t+7)', fontweight='bold', fontsize=13)

# ── Panel 1: NSE by lead time ─────────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(range(8), lead_nse, 'o-', color='steelblue', lw=2)
ax1.axhline(0, color='gray', lw=0.8, ls='--')
ax1.fill_between(range(8), 0, lead_nse, alpha=0.15, color='steelblue')
ax1.set_xlabel('Lead time (days)')
ax1.set_ylabel('NSE')
ax1.set_title('NSE degradation by lead time')
ax1.set_xticks(range(8))
ax1.grid(True, alpha=0.3)

# ── Panel 2: MAE and bias by lead time ────────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(range(8), lead_mae,  'o-', color='tomato',    lw=2, label='MAE')
ax2.plot(range(8), lead_bias, 's--', color='darkorange', lw=1.5, label='Bias')
ax2.axhline(0, color='gray', lw=0.8)
ax2.set_xlabel('Lead time (days)')
ax2.set_ylabel('m³/s')
ax2.set_title('MAE and bias by lead time')
ax2.set_xticks(range(8))
ax2.legend()
ax2.grid(True, alpha=0.3)

# ── Panel 3: Seasonal error distribution (violin) ─────────────────────────────
ax3 = fig.add_subplot(gs[0, 2])
df_seas = df_all[['err']].copy().dropna()
df_seas['month'] = df_seas.index.month
df_seas['season'] = pd.cut(df_seas['month'], bins=[0,3,6,9,12],
                           labels=['Jan–Mar', 'Apr–Jun', 'Jul–Sep', 'Oct–Dec'])
season_groups = [df_seas[df_seas['season'] == s]['err'].values
                 for s in ['Jan–Mar', 'Apr–Jun', 'Jul–Sep', 'Oct–Dec']]
vp = ax3.violinplot(season_groups, positions=[1,2,3,4], showmedians=True)
ax3.axhline(0, color='black', lw=0.8)
ax3.set_xticks([1,2,3,4])
ax3.set_xticklabels(['Jan–Mar', 'Apr–Jun', 'Jul–Sep', 'Oct–Dec'], fontsize=9)
ax3.set_ylabel('Error t+7 (m³/s)')
ax3.set_title('Error distribution by season')
ax3.grid(True, axis='y', alpha=0.3)

# ── Panel 4: Residual time series (full test period) ─────────────────────────
ax4 = fig.add_subplot(gs[1, :])
err_clean = np.where(np.isnan(err_t7), 0, err_t7)
ax4.fill_between(dates, 0, err_clean,
                 where=err_clean > 0, alpha=0.5, color='tomato',    label='Overestimate')
ax4.fill_between(dates, 0, err_clean,
                 where=err_clean < 0, alpha=0.5, color='steelblue', label='Underestimate')
# Mark flood events
for ev_name, peak_dt in [('Nov/2008', '2008-11-23'), ('Set/2011', '2011-09-11')]:
    ax4.axvline(pd.Timestamp(peak_dt), color='black', lw=1.2, ls=':', alpha=0.8)
    ax4.text(pd.Timestamp(peak_dt), ax4.get_ylim()[1] if ax4.get_ylim()[1] != 1 else 500,
             f' {ev_name}', fontsize=8, va='top')
ax4.axhline(0, color='black', lw=0.8)
ax4.set_ylabel('Error t+7 (m³/s)')
ax4.set_title('Residual time series (2008–2024): positive = overestimate, negative = underestimate')
ax4.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax4.xaxis.set_major_locator(mdates.YearLocator())
ax4.legend(loc='upper right')
ax4.grid(True, alpha=0.2)

# ── Panel 5: ACF of residuals (t+7) ──────────────────────────────────────────
ax5 = fig.add_subplot(gs[2, 0:2])
err_filled = pd.Series(err_t7, index=dates).interpolate().fillna(0).values
plot_acf(err_filled, lags=60, ax=ax5, alpha=0.05)
ax5.set_xlabel('Lag (days)')
ax5.set_ylabel('Autocorrelation')
ax5.set_title('ACF of t+7 residuals\n(significant lags → systematic timing error)')
ax5.grid(True, alpha=0.3)

# ── Panel 6: Monthly median bias ─────────────────────────────────────────────
ax6 = fig.add_subplot(gs[2, 2])
df_all_clean = df_all[['err']].dropna().copy()
monthly_bias = df_all_clean.groupby(df_all_clean.index.month)['err'].median()
colors = ['tomato' if v > 0 else 'steelblue' for v in monthly_bias.values]
ax6.bar(monthly_bias.index, monthly_bias.values, color=colors, edgecolor='gray', lw=0.5)
ax6.axhline(0, color='black', lw=0.8)
ax6.set_xticks(range(1, 13))
ax6.set_xticklabels(['J','F','M','A','M','J','J','A','S','O','N','D'], fontsize=9)
ax6.set_xlabel('Month')
ax6.set_ylabel('Median error t+7 (m³/s)')
ax6.set_title('Monthly median bias')
ax6.grid(True, axis='y', alpha=0.3)

fig.savefig(FIGURES / 'diag4_residual_structure.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Summary: Diagnostic Findings

Run all cells above, then fill in observed values below.

In [ ]:
# ── Automated summary report ──────────────────────────────────────────────────
nse_t0 = lead_nse[0]
nse_t7 = lead_nse[7]
nse_drop = nse_t0 - nse_t7

best_lag = lags[np.argmax(np.abs([corrs[l]['r'] for l in lags]))]
best_r   = corrs[best_lag]['r']

bias_below_600 = stats_df.loc[stats_df['obs_mean'] < 600, 'rel_bias'].mean()
bias_above_600 = stats_df.loc[stats_df['obs_mean'] >= 600, 'rel_bias'].mean()

worst_season = df_all[['err']].dropna().copy()
worst_season['month'] = worst_season.index.month
monthly_mae = worst_season.groupby('month')['err'].apply(lambda x: np.abs(x).mean())
worst_month = monthly_mae.idxmax()
month_names = {1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'May',6:'Jun',
               7:'Jul',8:'Aug',9:'Sep',10:'Oct',11:'Nov',12:'Dec'}

print('=' * 60)
print('PHASE 1 DIAGNOSTIC SUMMARY')
print('=' * 60)
print()
print('1. RESIDUAL ANALYSIS BY QUANTILE')
print(f'   Relative bias below ~600 m³/s:  {bias_below_600:+.1f}%')
print(f'   Relative bias above ~600 m³/s:  {bias_above_600:+.1f}%')
print(f'   Gap:                            {bias_above_600 - bias_below_600:+.1f} pp')
print(f'   → {"Sudden cliff" if abs(bias_above_600-bias_below_600)>20 else "Gradual degradation"} above training domain')
print()
print('2. ANTECEDENT PRECIP LAG CORRELATION')
print(f'   Best lag (highest |r|):          {best_lag} days (r={best_r:.3f})')
print(f'   Sep/2011 antecedent 60d precip:  {val_2011:.0f} mm  (P{pct_2011:.0f})')
print(f'   → Recommended seq_length for exp06: ≥{best_lag} days')
print()
print('3. CHIRPS PRECIPITATION (see diag3_chirps_bias_events.png)')
print('   Check ratio column above: < 1 = CHIRPS underestimates forcing')
print()
print('4. RESIDUAL TEMPORAL STRUCTURE')
print(f'   NSE t+0 → t+7:  {nse_t0:.3f} → {nse_t7:.3f}  (drop: {nse_drop:.3f})')
print(f'   Worst month by MAE: {month_names[worst_month]} (MAE={monthly_mae[worst_month]:.1f} m³/s)')
print(f'   ACF: check plot — significant autocorrelation at short lags indicates')
print(f'        systematic timing errors correctable by increasing seq_length')
print()
print('=' * 60)
print('RECOMMENDED EXPERIMENTS')
print('=' * 60)
print(f'  exp06: seq_length={max(best_lag, 30)} (primary fix for antecedent moisture gap)')
print(f'  exp07: seq_length={max(best_lag, 30)} + ERA5 soil moisture as dynamic feature')
print(f'  exp08: CMAL head (fixes saturation above training max)')
print(f'  exp09: exp06 + exp08 combined')